In [3]:
import pandas as pd
from pathlib import Path
import xml.etree.ElementTree as ET

In [4]:

BASE_DIR = Path.cwd().parent
RAW_DIR = BASE_DIR / "data" / "raw"

transactions_path = RAW_DIR / "transactions_2026.csv"
clients_path = RAW_DIR / "client.xml"
products_csv_path = RAW_DIR / "product.csv"
products_xml_path = RAW_DIR / "product.xml"

In [5]:
transactions = pd.read_csv(transactions_path)
products_csv = pd.read_csv(products_csv_path)

In [6]:
clients = pd.read_xml(clients_path)
products_xml = pd.read_xml(products_xml_path)

In [7]:
datasets = {
    "transactions": transactions,
    "clients": clients,
    "products_csv": products_csv,
    "products_xml": products_xml,
}

for name, df in datasets.items():
    print(f"\n--- {name.upper()} ---")
    print("Dimensions :", df.shape)
    print("Colonnes :", df.columns.tolist())
    print(df.head())


--- TRANSACTIONS ---
Dimensions : (10343, 9)
Colonnes : ['trsx_id', 'client_id', 'cart_id', 'product_id', 'date', 'amount', 'price', 'products_price', 'cart_price']
     trsx_id client_id   cart_id product_id                 date  amount  \
0  TRX000001   CLI0349  CRT00519    PRD0152  2026-04-12 07:24:58      10   
1  TRX000002   CLI0335  CRT00992    PRD0050  2026-03-31 01:20:13       5   
2  TRX000003   CLI0246  CRT00421    PRD0176  2026-04-29 16:12:53       8   
3  TRX000004   CLI0249  CRT01364    PRD0059  2026-01-14 15:52:09       2   
4  TRX000005   CLI0057  CRT02032    PRD0050  2026-07-07 02:10:57       6   

    price  products_price  cart_price  
0  336.37         3363.70     4507.13  
1  164.79          823.95     1606.03  
2  288.19         2305.52     6573.80  
3  252.15          504.30     5839.25  
4  164.79          988.74      988.74  

--- CLIENTS ---
Dimensions : (435, 6)
Colonnes : ['client_id', 'name', 'age', 'sexe', 'opening_date', 'status']
  client_id           na

In [8]:
def audit_dataframe(df: pd.DataFrame, name: str) -> pd.DataFrame:
    audit = pd.DataFrame({
        "dataset": name,
        "colonne": df.columns,
        "type": df.dtypes.astype(str).values,
        "nb_lignes": len(df),
        "nb_valeurs_nulles": df.isna().sum().values,
        "taux_valeurs_nulles_pct": (
            df.isna().mean().mul(100).round(2).values
        ),
        "nb_valeurs_uniques": df.nunique(dropna=False).values,
    })

    return audit

In [9]:
audit_global = pd.concat(
    [
        audit_dataframe(df, name)
        for name, df in datasets.items()
    ],
    ignore_index=True
)

audit_global

,dataset,colonne,type,nb_lignes,nb_valeurs_nulles,taux_valeurs_nulles_pct,nb_valeurs_uniques
0,transactions,trsx_id,object,10343,0,0.0,10343
1,transactions,client_id,object,10343,0,0.0,435
2,transactions,cart_id,object,10343,0,0.0,3333
3,transactions,product_id,object,10343,0,0.0,200
4,transactions,date,object,10343,0,0.0,3333
5,transactions,amount,int64,10343,0,0.0,10
6,transactions,price,float64,10343,0,0.0,199
7,transactions,products_price,float64,10343,0,0.0,1962
8,transactions,cart_price,float64,10343,0,0.0,3247
9,clients,client_id,object,435,0,0.0,435


In [10]:
for name, df in datasets.items():
    print(
        name,
        "- doublons complets :",
        df.duplicated().sum()
    )

transactions - doublons complets : 0
clients - doublons complets : 0
products_csv - doublons complets : 0
products_xml - doublons complets : 0


In [11]:
print(
    "Client ID dupliqués :",
    clients["client_id"].duplicated().sum()
)

print(
    "Product ID CSV dupliqués :",
    products_csv["product_id"].duplicated().sum()
)

print(
    "Product ID XML dupliqués :",
    products_xml["product_id"].duplicated().sum()
)

Client ID dupliqués : 0
Product ID CSV dupliqués : 0
Product ID XML dupliqués : 0


In [37]:
products_all = pd.concat(
    [products_csv, products_xml],
    ignore_index=True
)

print("Nombre total de produits :", len(products))
print(
    "Product ID dupliqués après fusion :",
    products["product_id"].duplicated().sum()
)

Nombre total de produits : 200
Product ID dupliqués après fusion : 0


In [13]:
clients_inconnus = transactions.loc[
    ~transactions["client_id"].isin(clients["client_id"]),
    "client_id"
].unique()

produits_inconnus = transactions.loc[
    ~transactions["product_id"].isin(products["product_id"]),
    "product_id"
].unique()

print("Clients inconnus :", clients_inconnus)
print("Produits inconnus :", produits_inconnus)

Clients inconnus : []
Produits inconnus : []


In [15]:
transactions.groupby("cart_id")["product_id"].nunique().describe()

count    3333.000000
mean        3.083108
std         1.601318
min         1.000000
25%         2.000000
50%         3.000000
75%         4.000000
max        10.000000
Name: product_id, dtype: float64

In [16]:
transactions.groupby("cart_id")["product_id"].nunique().sort_values(
    ascending=False
).head(10)

cart_id
CRT03053    10
CRT00620     9
CRT00909     9
CRT00060     9
CRT01299     9
CRT01181     9
CRT02000     9
CRT02495     9
CRT00767     9
CRT01115     9
Name: product_id, dtype: int64

In [17]:
transactions["trsx_id"].nunique() == len(transactions)

True

In [18]:
produits_par_panier = transactions.groupby("cart_id")["product_id"].nunique()

print(produits_par_panier.describe())
print(
    "Nombre de paniers avec plusieurs produits :",
    (produits_par_panier > 1).sum()
)

count    3333.000000
mean        3.083108
std         1.601318
min         1.000000
25%         2.000000
50%         3.000000
75%         4.000000
max        10.000000
Name: product_id, dtype: float64
Nombre de paniers avec plusieurs produits : 2778


In [19]:
doublons_panier_produit = transactions.duplicated(
    subset=["cart_id", "product_id"],
    keep=False
)

transactions.loc[doublons_panier_produit].sort_values(
    ["cart_id", "product_id"]
)

,trsx_id,client_id,cart_id,product_id,date,amount,price,products_price,cart_price
3737,TRX003738,CLI0421,CRT00073,PRD0158,2026-01-15 18:45:37,10,12.47,124.70,3415.43
4861,TRX004862,CLI0421,CRT00073,PRD0158,2026-01-15 18:45:37,5,12.47,62.35,3415.43
1997,TRX001998,CLI0146,CRT00089,PRD0027,2026-01-08 14:04:27,5,209.57,1047.85,3674.03
10214,TRX010215,CLI0146,CRT00089,PRD0027,2026-01-08 14:04:27,2,209.57,419.14,3674.03
1807,TRX001808,CLI0272,CRT00139,PRD0048,2026-03-18 15:02:26,5,275.16,1375.80,3301.92
...,...,...,...,...,...,...,...,...,...
6889,TRX006890,CLI0126,CRT03271,PRD0040,2026-01-19 07:50:04,8,256.15,2049.20,8073.75
1735,TRX001736,CLI0352,CRT03401,PRD0143,2026-05-13 23:17:39,6,331.85,1991.10,5926.94
8317,TRX008318,CLI0352,CRT03401,PRD0143,2026-05-13 23:17:39,8,331.85,2654.80,5926.94
422,TRX000423,CLI0183,CRT03404,PRD0102,2026-05-08 01:07:30,1,233.03,233.03,6769.02


In [20]:
transactions.duplicated(
    subset=["cart_id", "product_id"]
).sum()

np.int64(67)

In [21]:
clients_par_panier = transactions.groupby("cart_id")["client_id"].nunique()

print(clients_par_panier.describe())
print(
    "Paniers associés à plusieurs clients :",
    (clients_par_panier > 1).sum()
)

count    3333.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: client_id, dtype: float64
Paniers associés à plusieurs clients : 0


In [22]:
dates_par_panier = transactions.groupby("cart_id")["date"].nunique()

print(
    "Paniers associés à plusieurs dates :",
    (dates_par_panier > 1).sum()
)

Paniers associés à plusieurs dates : 0


In [23]:
import numpy as np

coherence_ligne = np.isclose(
    transactions["products_price"],
    transactions["amount"] * transactions["price"],
    rtol=1e-5,
    atol=0.01
)

print(
    "Lignes avec un montant incohérent :",
    (~coherence_ligne).sum()
)

Lignes avec un montant incohérent : 0


In [24]:
totaux_calcules = (
    transactions.groupby("cart_id")["products_price"]
    .sum()
    .round(2)
)

totaux_enregistres = (
    transactions.groupby("cart_id")["cart_price"]
    .first()
    .round(2)
)

comparaison_paniers = (
    totaux_calcules
    .rename("total_calcule")
    .to_frame()
    .join(totaux_enregistres.rename("total_enregistre"))
)

comparaison_paniers["ecart"] = (
    comparaison_paniers["total_calcule"]
    - comparaison_paniers["total_enregistre"]
).round(2)

comparaison_paniers[
    comparaison_paniers["ecart"].abs() > 0.01
]

,total_calcule,total_enregistre,ecart
cart_id,,,


In [25]:
print(
    "Paniers avec un total incohérent :",
    (comparaison_paniers["ecart"].abs() > 0.01).sum()
)

Paniers avec un total incohérent : 0


In [26]:
products_csv.columns.tolist()

['product_id', 'product_name', 'product_price', 'price_date']

In [27]:
products_xml.columns.tolist()

['product_id', 'product_name', 'product_price', 'price_date']

In [28]:
products_all = pd.concat(
    [products_csv, products_xml],
    ignore_index=True
)

historique_prix = (
    products_all.groupby("product_id")
    .agg(
        nombre_lignes=("product_id", "size"),
        nombre_prix=("product_price", "nunique"),
        nombre_dates=("price_date", "nunique")
    )
)

historique_prix.sort_values(
    ["nombre_lignes", "nombre_prix", "nombre_dates"],
    ascending=False
).head(20)

,nombre_lignes,nombre_prix,nombre_dates
product_id,,,
PRD0001,1,1,1
PRD0002,1,1,1
PRD0003,1,1,1
PRD0004,1,1,1
PRD0005,1,1,1
PRD0006,1,1,1
PRD0007,1,1,1
PRD0008,1,1,1
PRD0009,1,1,1


In [29]:
print(
    "Produits avec plusieurs enregistrements :",
    (historique_prix["nombre_lignes"] > 1).sum()
)

print(
    "Produits avec plusieurs prix :",
    (historique_prix["nombre_prix"] > 1).sum()
)

print(
    "Produits avec plusieurs dates de prix :",
    (historique_prix["nombre_dates"] > 1).sum()
)

Produits avec plusieurs enregistrements : 0
Produits avec plusieurs prix : 0
Produits avec plusieurs dates de prix : 0


### pour dico

In [30]:
print("CLIENT")
print(clients.dtypes)
print()

print("PRODUITS CSV")
print(products_csv.dtypes)
print()

print("PRODUITS XML")
print(products_xml.dtypes)
print()

print("TRANSACTIONS")
print(transactions.dtypes)

CLIENT
client_id       object
name            object
age              int64
sexe            object
opening_date    object
status          object
dtype: object

PRODUITS CSV
product_id        object
product_name      object
product_price    float64
price_date        object
dtype: object

PRODUITS XML
product_id        object
product_name      object
product_price    float64
price_date        object
dtype: object

TRANSACTIONS
trsx_id            object
client_id          object
cart_id            object
product_id         object
date               object
amount              int64
price             float64
products_price    float64
cart_price        float64
dtype: object


In [31]:
clients["opening_date"] = pd.to_datetime(
    clients["opening_date"],
    errors="coerce"
)

products_csv["price_date"] = pd.to_datetime(
    products_csv["price_date"],
    errors="coerce"
)

products_xml["price_date"] = pd.to_datetime(
    products_xml["price_date"],
    errors="coerce"
)

transactions["date"] = pd.to_datetime(
    transactions["date"],
    errors="coerce"
)

In [32]:
print("Dates client invalides :", clients["opening_date"].isna().sum())
print("Dates produits CSV invalides :", products_csv["price_date"].isna().sum())
print("Dates produits XML invalides :", products_xml["price_date"].isna().sum())
print("Dates transactions invalides :", transactions["date"].isna().sum())

Dates client invalides : 0
Dates produits CSV invalides : 0
Dates produits XML invalides : 0
Dates transactions invalides : 0


In [33]:
print(clients["sexe"].value_counts(dropna=False))
print(clients["status"].value_counts(dropna=False))

sexe
F    229
M    206
Name: count, dtype: int64
status
active      389
inactive     46
Name: count, dtype: int64


In [34]:
print("Âges invalides :", (~clients["age"].between(0, 120)).sum())

print(
    "Quantités nulles ou négatives :",
    (transactions["amount"] <= 0).sum()
)

print(
    "Prix de transaction nuls ou négatifs :",
    (transactions["price"] <= 0).sum()
)

print(
    "Montants de ligne négatifs :",
    (transactions["products_price"] < 0).sum()
)

print(
    "Totaux de panier négatifs :",
    (transactions["cart_price"] < 0).sum()
)

print(
    "Prix produits nuls ou négatifs :",
    (products_all["product_price"] <= 0).sum()
)

Âges invalides : 0
Quantités nulles ou négatives : 0
Prix de transaction nuls ou négatifs : 0
Montants de ligne négatifs : 0
Totaux de panier négatifs : 0
Prix produits nuls ou négatifs : 0


In [35]:
sexes_valides = {"F", "M"}
statuts_valides = {"active", "inactive"}

print(
    "Sexes invalides :",
    (~clients["sexe"].isin(sexes_valides)).sum()
)

print(
    "Statuts invalides :",
    (~clients["status"].isin(statuts_valides)).sum()
)

Sexes invalides : 0
Statuts invalides : 0


In [36]:
print(
    "client_id au format invalide :",
    (~clients["client_id"].str.fullmatch(r"CLI\d{4}")).sum()
)

print(
    "product_id au format invalide :",
    (~products_all["product_id"].str.fullmatch(r"PRD\d{4}")).sum()
)

print(
    "trsx_id au format invalide :",
    (~transactions["trsx_id"].str.fullmatch(r"TRX\d{6}")).sum()
)

print(
    "cart_id au format invalide :",
    (~transactions["cart_id"].str.fullmatch(r"CRT\d{5}")).sum()
)

client_id au format invalide : 0
product_id au format invalide : 0
trsx_id au format invalide : 0
cart_id au format invalide : 0
